In [ ]:
# Statistical Significance Testing (Wilcoxon) for protocol C

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import joblib
from scipy.stats import wilcoxon
from sklearn.metrics import matthews_corrcoef
from sklearn.base import clone

RANDOM_STATE = 42
PROJECT = '/content/drive/MyDrive/AgileML/pipeline_v2'
IN1, IN2, IN3 = (f'{PROJECT}/01_data_prep', f'{PROJECT}/02_benchmark',
                 f'{PROJECT}/03_optimization')
OUT = f'{PROJECT}/04_statistical_testing'
os.makedirs(OUT, exist_ok=True)

X_train, X_test, y_train, y_test = joblib.load(f'{IN2}/split_chronological.pkl')
cv_ts = joblib.load(f'{IN2}/cv_scheme.pkl')
rezultate_base = joblib.load(f'{IN2}/rezultate_base.pkl')
modele_opt = joblib.load(f'{IN3}/modele_optimizate.pkl')
thresholds = joblib.load(f'{IN3}/thresholds.pkl')
MODEL_FINAL = joblib.load(f'{IN3}/model_final_name.pkl')
RUNNER_UP   = joblib.load(f'{IN3}/runner_up_name.pkl')

print(f'Final model: {MODEL_FINAL} | Runner-up: {RUNNER_UP}')

Mounted at /content/drive


/usr/lib/python3.13/pickle.py:1754: UserWarning: [07:02:20] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


Final model: Random Forest | Runner-up: XGBoost


In [ ]:
# Calculate MCC for top 2 optimized models and for the first model optimized and non-optimized


def mcc_per_fold(model, X, y, cv, threshold=0.5):
    out = []
    for tr, va in cv.split(X):
        m = clone(model)
        m.fit(X.iloc[tr], y.iloc[tr])
        proba = m.predict_proba(X.iloc[va])[:, 1]
        out.append(matthews_corrcoef(y.iloc[va], (proba >= threshold).astype(int)))
    return np.array(out)

print(f' MCC per fold — {MODEL_FINAL} (optimized) ...')
mcc_final = mcc_per_fold(modele_opt[MODEL_FINAL], X_train, y_train, cv_ts,
                         thresholds[MODEL_FINAL])

print(f' MCC per fold — {RUNNER_UP} (optimized) ...')
mcc_runner = mcc_per_fold(modele_opt[RUNNER_UP], X_train, y_train, cv_ts,
                          thresholds[RUNNER_UP])

print(f' MCC per fold — {MODEL_FINAL} (non-optimized baseline) ...')
mcc_own_base = mcc_per_fold(rezultate_base[MODEL_FINAL]['model'], X_train, y_train,
                            cv_ts, 0.5)

print(f'\n{MODEL_FINAL} (opt.):   {mcc_final.round(3)}')
print(f'{RUNNER_UP} (opt.):   {mcc_runner.round(3)}')
print(f'{MODEL_FINAL} (base):  {mcc_own_base.round(3)}')

 MCC per fold — Random Forest (optimized) ...
 MCC per fold — XGBoost (optimized) ...
 MCC per fold — Random Forest (non-optimized baseline) ...

Random Forest (opt.):   [0.109 0.171 0.162 0.126 0.219 0.329 0.282 0.196 0.327 0.214]
XGBoost (opt.):   [0.098 0.097 0.158 0.083 0.17  0.327 0.253 0.222 0.314 0.224]
Random Forest (base):  [0.126 0.213 0.188 0.123 0.193 0.303 0.278 0.178 0.195 0.192]


In [ ]:
# Wilcoxon signed-rank test

s1, p1 = wilcoxon(mcc_final, mcc_runner)
s2, p2 = wilcoxon(mcc_final, mcc_own_base)

df_wil = pd.DataFrame([
    {'Comparison': f'{MODEL_FINAL} (opt.) vs {RUNNER_UP} (opt.)',
     'Mean MCC (A)': round(mcc_final.mean(), 3), 'Mean MCC (B)': round(mcc_runner.mean(), 3),
     'Statistic': round(s1, 3), 'p-value': round(p1, 4),
     'Significant (a=0.05)': 'Yes' if p1 < 0.05 else 'No'},
    {'Comparison': f'{MODEL_FINAL} (opt.) vs {MODEL_FINAL} (base)',
     'Mean MCC (A)': round(mcc_final.mean(), 3), 'Mean MCC (B)': round(mcc_own_base.mean(), 3),
     'Statistic': round(s2, 3), 'p-value': round(p2, 4),
     'Significant (a=0.05)': 'Yes' if p2 < 0.05 else 'No'},
])
df_wil.to_csv(f'{OUT}/table_wilcoxon.csv', index=False)
print(df_wil.to_string(index=False))

                                  Comparison  Mean MCC (A)  Mean MCC (B)  Statistic  p-value Significant (a=0.05)
      Random Forest (opt.) vs XGBoost (opt.)         0.214         0.195        9.0   0.0645                   No
Random Forest (opt.) vs Random Forest (base)         0.214         0.199       18.0   0.3750                   No


In [ ]:
print('Saved to', OUT)

Saved to /content/drive/MyDrive/AgileML/pipeline_v2/04_statistical_testing
